In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import random
from collections import deque
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# =============================================
# 1. LOAD DATA (TRAIN ONLY)
# =============================================
state_data = pd.read_csv("../data/state_dataset.csv", index_col="Date", parse_dates=True)
market_data = pd.read_csv("../data/market_master.csv", index_col="Date", parse_dates=True)

data = state_data.join(market_data[["nifty_ret"]], how="inner")
train_data = data.loc["2019":"2022"].copy()

print(f"Training: {train_data.index[0].date()} → {train_data.index[-1].date()}")

# =============================================
# 2. ADVANCED REWARD FUNCTION
# =============================================
class RewardFunction:
    def __init__(self):
        self.tc = 0.0005
        
        # tuned parameters
        self.risk_penalty = 0.12
        self.dd_penalty = 0.35
        self.vol_penalty = 0.08
        self.flat_bonus = 0.015
        self.trade_penalty = 0.002  # NEW
    
    def compute(self, net_ret, prev_pos, new_pos, drawdown, vol_signal):
        
        reward = net_ret
        
        # Transaction cost
        reward -= abs(new_pos - prev_pos) * self.tc
        
        # Penalize overtrading
        if new_pos != prev_pos:
            reward -= self.trade_penalty
        
        # Risk penalty (quadratic)
        reward -= self.risk_penalty * (net_ret ** 2)
        
        # Drawdown penalty (strong)
        if drawdown < -0.03:
            reward -= self.dd_penalty * abs(drawdown)
        
        # Volatility penalty
        if vol_signal > 0.5:
            reward -= self.vol_penalty * vol_signal
        
        # Flat bonus (smart inactivity)
        if new_pos == 0 and abs(net_ret) < 0.004:
            reward += self.flat_bonus
        
        return reward

# =============================================
# 3. ENVIRONMENT (CONSISTENT)
# =============================================
class QuantumAlphaEnv:
    def __init__(self, data):
        self.data = data.reset_index(drop=True)
        self.feature_cols = ["mom_20_norm", "vol_signal_norm", "trend_signal_norm",
                             "dd_signal_norm", "vix_signal_norm", "breadth_signal_norm"]
        self.reward_fn = RewardFunction()
        self.max_steps = len(self.data) - 1
        self.reset()
    
    def reset(self):
        self.current_step = 0
        self.position = 0
        self.balance = 1.0
        self.peak = 1.0
        return self._get_observation()
    
    def _get_observation(self):
        row = self.data.iloc[self.current_step]
        obs = row[self.feature_cols].values.astype(np.float32)
        return np.append(obs, self.position)
    
    def step(self, action):
        prev_pos = self.position
        new_pos = {0:0, 1:1, 2:-1}[action]
        
        ret = self.data.iloc[self.current_step]["nifty_ret"]
        cost = abs(new_pos - prev_pos) * 0.0005
        
        net_ret = prev_pos * ret - cost
        
        self.balance *= (1 + net_ret)
        self.peak = max(self.peak, self.balance)
        
        drawdown = (self.balance - self.peak) / self.peak
        vol_signal = self.data.iloc[self.current_step]["vol_signal_norm"]
        
        reward = self.reward_fn.compute(net_ret, prev_pos, new_pos, drawdown, vol_signal)
        
        self.position = new_pos
        self.current_step += 1
        
        done = self.current_step >= self.max_steps
        
        return self._get_observation(), reward, done, {"net_ret": net_ret}

# =============================================
# 4. MODEL
# =============================================
class DQN(nn.Module):
    def __init__(self, state_size=7, action_size=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_size, 128), nn.ReLU(),
            nn.Linear(128, 128), nn.ReLU(),
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64, action_size)
        )
    
    def forward(self, x):
        return self.net(x)

# =============================================
# 5. TRAINING SETUP
# =============================================
env = QuantumAlphaEnv(train_data)

policy_net = DQN().to(device)
target_net = DQN().to(device)
target_net.load_state_dict(policy_net.state_dict())

optimizer = optim.Adam(policy_net.parameters(), lr=2e-4)
memory = deque(maxlen=100000)

gamma = 0.99
epsilon = 0.4
epsilon_min = 0.05
epsilon_decay = 0.995

batch_size = 128
target_update = 200

# =============================================
# 6. TRAIN LOOP (STABLE VERSION)
# =============================================
episodes = 180
episode_rewards = []

print("\nTraining improved agent...\n")

for ep in range(episodes):
    
    state = env.reset()
    total_reward = 0
    
    while True:
        
        if random.random() < epsilon:
            action = random.randint(0, 2)
        else:
            with torch.no_grad():
                s = torch.FloatTensor(state).unsqueeze(0).to(device)
                action = policy_net(s).argmax().item()
        
        next_state, reward, done, _ = env.step(action)
        
        memory.append((state, action, reward, next_state, done))
        state = next_state
        total_reward += reward
        
        if len(memory) > batch_size:
            batch = random.sample(memory, batch_size)
            states, actions, rewards, next_states, dones = zip(*batch)
            
            states = torch.FloatTensor(np.array(states)).to(device)
            actions = torch.LongTensor(actions).to(device)
            rewards = torch.FloatTensor(rewards).to(device)
            next_states = torch.FloatTensor(np.array(next_states)).to(device)
            dones = torch.FloatTensor(dones).to(device)
            
            current_q = policy_net(states).gather(1, actions.unsqueeze(1)).squeeze()
            next_q = target_net(next_states).max(1)[0]
            target_q = rewards + gamma * next_q * (1 - dones)
            
            loss = nn.SmoothL1Loss()(current_q, target_q)
            
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(policy_net.parameters(), 1.0)
            optimizer.step()
        
        if done:
            break
    
    if ep % target_update == 0:
        target_net.load_state_dict(policy_net.state_dict())
    
    epsilon = max(epsilon_min, epsilon * epsilon_decay)
    episode_rewards.append(total_reward)
    
    if ep % 20 == 0:
        print(f"Episode {ep} | Reward: {total_reward:.2f}")

# =============================================
# 7. SAVE MODEL
# =============================================
torch.save(policy_net.state_dict(), "../models/dqn_final_reward_optimized.pth")

print("\n Optimized model saved!")

# =============================================
# 8. REWARD PLOT
# =============================================
plt.figure(figsize=(10,4))
plt.plot(episode_rewards)
plt.title("Training Rewards (Optimized Reward)")
plt.grid(True)
plt.show()